# CBAM ResNet18 with NCA and kNN Classification

This model leverages the powerful feature extraction capabilities of a fine-tuned CBAM-enhanced ResNet18. The traditional CNN classification head is replaced by an architecture similar to the ExMPLPQ model: extracted features are reduced using NCA and classified by kNN.

## 1. Import Required Libraries


In [ ]:
# Data handling and visualization
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Utility
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
from sklearn.neighbors import NeighborhoodComponentsAnalysis

# For Kaggle dataset download
import kagglehub


## 2. Download and Prepare MRI Dataset

Download the MRI dataset from Kaggle using kagglehub, set up the directory structure, and verify available categories.

In [ ]:
# Set custom download directory
relative_path = "../../../data/raw"
os.environ["KAGGLEHUB_CACHE"] = relative_path

# Download dataset if not already present
if not os.listdir(relative_path):
    path = kagglehub.dataset_download("buraktaci/multiple-sclerosis")
    print("Dataset downloaded to:", path)
else:
    print(f"Dataset already exists in {relative_path}")
    path = os.path.join(relative_path, "datasets/buraktaci/multiple-sclerosis/versions/1/MS/")

# List available categories
categories = os.listdir(path)
print("Available categories:", categories)

## 3. Visualize Sample MRI Images

Display random samples of control and MS MRI images for visual inspection.

In [ ]:
# Define image categories
control_axial = "Control Axial_crop"
ms_axial = "MS Axial_crop"

classes = [control_axial, ms_axial]

# Visualize 3 random images from each category
for cat in classes:
    image_dir = os.path.join(path, cat)
    images = os.listdir(image_dir)
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Sample images: {cat}', fontsize=18)
    for i in range(3):
        idx = np.random.randint(0, len(images))
        img = np.array(Image.open(os.path.join(image_dir, images[idx])))
        ax[i].imshow(img, cmap='gray')
        ax[i].axis('off')
    plt.show()

## 4. Preprocess MRI Images

Resize images to 224x224, normalize pixel values, encode labels, and optionally augment data. Prepare PyTorch datasets and dataloaders.

In [ ]:
IMG_SIZE = (224, 224)

class MRIDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label

# Gather image paths and labels
image_paths = []
labels = []
for label, cat in enumerate(classes):
    image_dir = os.path.join(path, cat)
    for fname in os.listdir(image_dir):
        img_path = os.path.join(image_dir, fname)
        image_paths.append(img_path)
        labels.append(label)

print(f"Total images: {len(image_paths)}")

# Define transforms
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

## 5. Split Data into Train and Test Sets

Split the dataset into training and testing sets using sklearn's train_test_split. Prepare corresponding dataloaders.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

# Create datasets
train_dataset = MRIDataset(X_train, y_train, transform=train_transform)
test_dataset = MRIDataset(X_test, y_test, transform=test_transform)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 6. Define CBAM ResNet18 Feature Extractor

Define the model to output the 512-dimensional feature vector, stopping just before the final classification head.

In [ ]:
### Define CBAMs 
# Channel Attention (like SE)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)


# Spatial Attention
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_cat)
        return self.sigmoid(out)


# CBAM Block = Channel + Spatial attention
class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_channels, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        out = x * self.ca(x)
        out = out * self.sa(out)
        return out

### CBAM-ResNet18 Feature Extractor
class CBAMResNet18(nn.Module):
    """
    Model designed to output the deep feature vector (512-dim), 
    acting as the feature generation step.
    """
    def __init__(self, pretrained=True):
        super(CBAMResNet18, self).__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        # Features: up to layer4, removing the final pooling and fc layer
        self.features = nn.Sequential(*list(resnet.children())[:-2]) 
        self.cbam = CBAM(in_channels=512) 
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 512

    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        # x is the 512-dimensional feature vector
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CBAMResNet18().to(device)
print(model)

# For training purposes, we wrap it with a temporary classifier head
# to perform standard fine-tuning first.
class MS_CBAM_ResNet18_Full(nn.Module):
    def __init__(self, num_classes=1):
        super(MS_CBAM_ResNet18_Full, self).__init__()
        self.backbone = CBAMResNet18(pretrained=True)
        self.classifier = nn.Sequential(
            nn.Linear(self.backbone.out_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x

# 7. Train the Feature Extractor
Train the full CNN model first to get a good feature representation. Save the entire model's weights and then extract the backbone weights later.

In [ ]:
# Instantiate the full model and loss function
full_model = MS_CBAM_ResNet18_Full().to(device)
criterion = nn.BCEWithLogitsLoss()

# --- 7(a). Train the Classifier Head (Freezing Backbone) ----

# Freeze feature extraction layers
for param in full_model.backbone.parameters():
    param.requires_grad = False

# Optimizer targets only the classifier head parameters
optimizer = optim.Adam(full_model.classifier.parameters(), lr=1e-3)
num_epochs = 10

# Initialize lists to track metrics for plotting
train_losses = []
train_accuracies = []

print("Starting Classifier Head Training...")
for epoch in range(num_epochs):
    full_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        # Ensure labels are float32 and unsqueezed for BCEWithLogitsLoss
        labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = full_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        
        # Calculate training accuracy
        preds = (torch.sigmoid(outputs).detach().cpu().numpy() > 0.5).astype(int)
        correct += (preds == labels.cpu().numpy()).sum()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    print(f"Head Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")

# Plot loss curve
plt.figure(figsize=(8,5))
plt.plot(range(1, num_epochs+1), train_losses, marker='o')
plt.title("Training Loss vs Epochs (Head Only)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.ylim(0, 1)
plt.grid(True)
plt.show()

# Save head-only weights (optional, but good practice)
os.makedirs("./saved-weights", exist_ok=True)
torch.save(full_model.state_dict(), "./saved-weights/cbam_variant_head_weights.pth")
print("Head-only weights saved.")


In [ ]:

# --- 7(b). Train Entire Model (Fine-tuning) ----

# Unfreeze backbone layers
for param in full_model.backbone.parameters():
    param.requires_grad = True

# Optimizer targets all model parameters with a lower learning rate
optimizer = optim.Adam(full_model.parameters(), lr=1e-5)
num_finetune_epochs = 5

print("\nStarting Full Model Fine-Tuning...")
for epoch in range(num_finetune_epochs):
    full_model.train()
    running_loss = 0.0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = full_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += labels.size(0)

    epoch_loss = running_loss / total
    print(f"Fine-tune Epoch {epoch+1}/{num_finetune_epochs} - Loss: {epoch_loss:.4f}")

# Save the full fine-tuned model weights
torch.save(full_model.state_dict(), "./saved-weights/cbam_variant_full_model.pth")
print("Full fine-tuned model weights saved.")

# 8. Feature Extraction, NCA, and kNN Classification
This section implements the new, adapted pipeline using the trained CNN as a feature extractor, followed by feature selection and kNN classification.

## 8.1 Load Feature Extractor and Extract Deep Features

In [ ]:
print("\n--- Starting Feature Extraction Pipeline ---")

# Load the trained full model weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
full_model = MS_CBAM_ResNet18_Full().to(device)
full_model.load_state_dict(torch.load("./saved-weights/cbam_variant_full_model.pth", map_location=device))

# Create the pure feature extractor and transfer the weights
feature_extractor = CBAMResNet18(pretrained=False).to(device)
feature_extractor.load_state_dict(full_model.backbone.state_dict())
feature_extractor.eval()
print("CBAM-ResNet18 Feature Extractor loaded successfully.")


def extract_features(data_loader, model):
    """Generates feature vectors and labels from a DataLoader using the model."""
    features = []
    labels = []
    # Note: We use the global y_train_labels and y_test_labels from Section 5 
    # for label consistency, though they are also available in the dataloader.
    with torch.no_grad():
        for images, batch_labels in data_loader:
            images = images.to(device)
            feature_vector = model(images)
            features.append(feature_vector.cpu().numpy())
            labels.append(batch_labels)
            
    return np.concatenate(features, axis=0), np.concatenate(labels, axis=0)

print("\nExtracting training features...")
X_train_features, y_train_features_labels = extract_features(train_loader, feature_extractor)

print("Extracting testing features...")
X_test_features, y_test_features_labels = extract_features(test_loader, feature_extractor)

print(f"Train Feature Shape: {X_train_features.shape}")
print(f"Test Feature Shape: {X_test_features.shape}")

## 8.2 Feature Selection using NCA (INCA Proxy)

In [58]:
# 512 features is a lot for a simple classifier
# NCA selects the most discriminative features for kNN.

TARGET_DIM = 256 # Reducing 512 features to 64 dimensions

nca = NeighborhoodComponentsAnalysis(
    n_components=TARGET_DIM, 
    random_state=42, 
    max_iter=500, # Increased iterations for better convergence
    tol=1e-5
)

print(f"\nFitting NCA to reduce 512 features to {TARGET_DIM}...")
# NCA is fitted ONLY on the training data
nca.fit(X_train_features, y_train_features_labels) 

# Transform both train and test features
X_train_selected = nca.transform(X_train_features)
X_test_selected = nca.transform(X_test_features)

print(f"Reduced Train Feature Shape: {X_train_selected.shape}")
print(f"Reduced Test Feature Shape: {X_test_selected.shape}")


Fitting NCA to reduce 512 features to 256...
Reduced Train Feature Shape: (1321, 256)
Reduced Test Feature Shape: (331, 256)


## 8.3 kNN Classification and Evaluation

In [59]:
# Train kNN on the selected features.
# The original paper used "Fine kNN", typically meaning k=1.
knn_classifier = KNeighborsClassifier(n_neighbors=20) 

print("\nTraining kNN classifier on NCA selected deep features...")
knn_classifier.fit(X_train_selected, y_train_features_labels)

# Predict on the test set
y_test_pred_knn = knn_classifier.predict(X_test_selected)

# Evaluate performance
print("\n--- kNN Classification Report on NCA Selected Features ---")
print("Accuracy:", accuracy_score(y_test_features_labels, y_test_pred_knn))
print("Precision:", precision_score(y_test_features_labels, y_test_pred_knn))
print("Recall:", recall_score(y_test_features_labels, y_test_pred_knn))
print("Confusion Matrix:\n", confusion_matrix(y_test_features_labels, y_test_pred_knn))
print("Classification Report:\n", classification_report(y_test_features_labels, y_test_pred_knn))


Training kNN classifier on NCA selected deep features...

--- kNN Classification Report on NCA Selected Features ---
Accuracy: 0.9274924471299094
Precision: 0.9491525423728814
Recall: 0.8615384615384616
Confusion Matrix:
 [[195   6]
 [ 18 112]]
Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.97      0.94       201
           1       0.95      0.86      0.90       130

    accuracy                           0.93       331
   macro avg       0.93      0.92      0.92       331
weighted avg       0.93      0.93      0.93       331

